In [1]:
import pandas as pd
import numpy as np
import os

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [2]:
patients = pd.read_csv("C:\\Users\\Acer\\Desktop\\Projects\\Hospital_Management\\Hospital_data\\patients.csv")
admissions = pd.read_csv("C:\\Users\\Acer\\Desktop\\Projects\\Hospital_Management\\Hospital_data\\admissions.csv")
diagnoses = pd.read_csv("C:\\Users\\Acer\\Desktop\\Projects\\Hospital_Management\\Hospital_data\\diagnoses.csv")
billing = pd.read_csv("C:\\Users\\Acer\\Desktop\\Projects\\Hospital_Management\\Hospital_data\\billing.csv")

In [3]:
datasets = {
    "patients": patients,
    "admissions": admissions,
    "diagnoses": diagnoses,
    "billing": billing
}

for name, df in datasets.items():
    print(f"\n{name.upper()}")
    print("Shape:", df.shape)
    print(df.head())
    print(df.info())


PATIENTS
Shape: (86400, 8)
                             patient_id  age gender          state  bpl_card  \
0  2b179f89-9af7-45cf-86b4-3e01a47a3b7b   36      F      Telangana     False   
1  ecaeb9f3-d263-4712-b9b6-3312af5626ca   56      F  Uttar Pradesh     False   
2  0160a5b6-877e-41e6-b84a-a2a51b367a36   67      F      Telangana     False   
3  216875fa-b49d-4c47-8c8d-7191ba34ac06   53      M        Haryana     False   
4  a2f3181d-e3f2-4fd3-8057-05a492ffcfb2   60      M    Maharashtra      True   

  insurance_type  comorbidity_count  prev_admissions  
0        Private                  0                0  
1            ESI                  0                0  
2            ESI                  3                0  
3       Ayushman                  3                0  
4       Ayushman                  4                1  
<class 'pandas.DataFrame'>
RangeIndex: 86400 entries, 0 to 86399
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype
---  ------       

In [4]:
for name, df in datasets.items():
    print(f"\n{name.upper()} - Missing Values")
    print(df.isnull().sum())


PATIENTS - Missing Values
patient_id               0
age                      0
gender                   0
state                    0
bpl_card                 0
insurance_type       24715
comorbidity_count        0
prev_admissions          0
dtype: int64

ADMISSIONS - Missing Values
admission_id      0
patient_id        0
admit_date        0
discharge_date    0
los_days          0
admit_type        0
ward_type         0
hospital_id       0
discharge_type    0
num_procedures    0
charlson_index    0
hba1c             0
creatinine        0
haemoglobin       0
systolic_bp       0
readmitted_30d    0
readmitted_7d     0
dtype: int64

DIAGNOSES - Missing Values
diag_id          0
admission_id     0
icd10_code       0
diag_desc        0
diag_rank        0
diag_category    0
dtype: int64

BILLING - Missing Values
bill_id              0
admission_id         0
total_cost_inr       0
govt_subsidy_inr     0
out_of_pocket_inr    0
cost_category        0
dtype: int64


In [5]:
for name, df in datasets.items():
    missing = df.isnull().sum()
    percentage = (missing / len(df)) * 100

    result = pd.DataFrame({
        "Missing_Count": missing,
        "Missing_Percentage": percentage
    })

    print(f"\n{name.upper()}")
    print(result[result["Missing_Count"] > 0])


PATIENTS
                Missing_Count  Missing_Percentage
insurance_type          24715           28.605324

ADMISSIONS
Empty DataFrame
Columns: [Missing_Count, Missing_Percentage]
Index: []

DIAGNOSES
Empty DataFrame
Columns: [Missing_Count, Missing_Percentage]
Index: []

BILLING
Empty DataFrame
Columns: [Missing_Count, Missing_Percentage]
Index: []


In [6]:
for name, df in datasets.items():
    print(name, "Duplicate Rows:", df.duplicated().sum())

patients Duplicate Rows: 0
admissions Duplicate Rows: 0
diagnoses Duplicate Rows: 0
billing Duplicate Rows: 0


In [7]:
print("Duplicate patient IDs:", patients["patient_id"].duplicated().sum())
print("Duplicate admission IDs:", admissions["admission_id"].duplicated().sum())
print("Duplicate diagnosis IDs:", diagnoses["diag_id"].duplicated().sum())
print("Duplicate bill IDs:", billing["bill_id"].duplicated().sum())

Duplicate patient IDs: 0
Duplicate admission IDs: 0
Duplicate diagnosis IDs: 0
Duplicate bill IDs: 0


In [8]:
admissions["admit_date"] = pd.to_datetime(
    admissions["admit_date"],
    dayfirst=True,
    errors="coerce"
)

admissions["discharge_date"] = pd.to_datetime(
    admissions["discharge_date"],
    dayfirst=True,
    errors="coerce"
)

In [9]:
print(admissions[["admit_date", "discharge_date"]].isnull().sum())

admit_date        0
discharge_date    0
dtype: int64


In [10]:
numeric_columns = [
    "age",
    "comorbidity_count",
    "prev_admissions"
]

for col in numeric_columns:
    patients[col] = pd.to_numeric(
        patients[col],
        errors="coerce"
    )

In [11]:
admission_numeric = [
    "los_days",
    "num_procedures",
    "charlson_index",
    "hba1c",
    "creatinine",
    "haemoglobin",
    "systolic_bp"
]

for col in admission_numeric:
    admissions[col] = pd.to_numeric(
        admissions[col],
        errors="coerce"
    )

In [12]:
print("Invalid ages:")
print(patients[(patients["age"] < 0) | (patients["age"] > 120)])

print("Invalid length of stay:")
print(admissions[admissions["los_days"] < 0])

print("Invalid billing:")
print(billing[
    (billing["total_cost_inr"] < 0) |
    (billing["out_of_pocket_inr"] < 0)
])

Invalid ages:
Empty DataFrame
Columns: [patient_id, age, gender, state, bpl_card, insurance_type, comorbidity_count, prev_admissions]
Index: []
Invalid length of stay:
Empty DataFrame
Columns: [admission_id, patient_id, admit_date, discharge_date, los_days, admit_type, ward_type, hospital_id, discharge_type, num_procedures, charlson_index, hba1c, creatinine, haemoglobin, systolic_bp, readmitted_30d, readmitted_7d]
Index: []
Invalid billing:
Empty DataFrame
Columns: [bill_id, admission_id, total_cost_inr, govt_subsidy_inr, out_of_pocket_inr, cost_category]
Index: []


In [13]:
billing["calculated_out_of_pocket"] = (
    billing["total_cost_inr"]
    - billing["govt_subsidy_inr"]
)

billing["billing_difference"] = (
    billing["out_of_pocket_inr"]
    - billing["calculated_out_of_pocket"]
)

print(
    billing[billing["billing_difference"] != 0]
)

Empty DataFrame
Columns: [bill_id, admission_id, total_cost_inr, govt_subsidy_inr, out_of_pocket_inr, cost_category, calculated_out_of_pocket, billing_difference]
Index: []


In [14]:
invalid_dates = admissions[
    admissions["discharge_date"] < admissions["admit_date"]
]

print(invalid_dates)

Empty DataFrame
Columns: [admission_id, patient_id, admit_date, discharge_date, los_days, admit_type, ward_type, hospital_id, discharge_type, num_procedures, charlson_index, hba1c, creatinine, haemoglobin, systolic_bp, readmitted_30d, readmitted_7d]
Index: []


In [15]:
missing_patients = admissions[
    ~admissions["patient_id"].isin(patients["patient_id"])
]

print(missing_patients)

Empty DataFrame
Columns: [admission_id, patient_id, admit_date, discharge_date, los_days, admit_type, ward_type, hospital_id, discharge_type, num_procedures, charlson_index, hba1c, creatinine, haemoglobin, systolic_bp, readmitted_30d, readmitted_7d]
Index: []


In [16]:
missing_admissions = billing[
    ~billing["admission_id"].isin(admissions["admission_id"])
]

print(missing_admissions)

Empty DataFrame
Columns: [bill_id, admission_id, total_cost_inr, govt_subsidy_inr, out_of_pocket_inr, cost_category, calculated_out_of_pocket, billing_difference]
Index: []


In [17]:
missing_diagnosis_admissions = diagnoses[
    ~diagnoses["admission_id"].isin(admissions["admission_id"])
]

print(missing_diagnosis_admissions)

Empty DataFrame
Columns: [diag_id, admission_id, icd10_code, diag_desc, diag_rank, diag_category]
Index: []


In [18]:
admissions["admit_year"] = admissions["admit_date"].dt.year
admissions["admit_month"] = admissions["admit_date"].dt.month
admissions["admit_month_name"] = admissions["admit_date"].dt.month_name()
admissions["admit_day"] = admissions["admit_date"].dt.day

In [19]:
admissions["calculated_los"] = (
    admissions["discharge_date"]
    - admissions["admit_date"]
).dt.days

In [21]:
text_columns = [
    "gender",
    "state",
    "insurance_type"
]

for col in text_columns:
    patients[col] = patients[col].astype("string").str.strip()

text_columns_admissions = [
    "admit_type",
    "ward_type",
    "discharge_type"
]

for col in text_columns_admissions:
    admissions[col] = admissions[col].astype("string").str.strip()

In [22]:
os.makedirs("cleaned_data", exist_ok=True)

patients.to_csv("cleaned_data/patients_cleaned.csv", index=False)
admissions.to_csv("cleaned_data/admissions_cleaned.csv", index=False)
diagnoses.to_csv("cleaned_data/diagnoses_cleaned.csv", index=False)
billing.to_csv("cleaned_data/billing_cleaned.csv", index=False)